# 09 - Model Evaluation & Fair Test Comparison

## 1. Held-Out Test Evaluation
In this notebook, all three trained models (Random Forest, XGBoost, and CatBoost) are evaluated on the held-out test partition (1,000 samples) under identical conditions.
Prioritized metrics:
- **Accuracy**
- **Precision & Recall**
- **F1-Score**
- **ROC-AUC**
- **PR-AUC (Average Precision)**
- **Confusion Matrix**

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import load_and_split_data
from src.features.preprocessing import create_full_pipeline
from src.models.train import get_candidate_models
from src.models.evaluation import evaluate_model, compare_models_metrics
from src.utils.plotting import plot_confusion_matrix, plot_roc_pr_curves
from src.config import RAW_DATA_PATH

X_train, X_test, y_train, y_test = load_and_split_data(RAW_DATA_PATH)
candidate_models = get_candidate_models()

## 2. Fit and Evaluate Candidates on Test Set

In [ ]:
eval_results = {}
pred_prob_dict = {}

for name, model in candidate_models.items():
    pipe = create_full_pipeline(model)
    pipe.fit(X_train, y_train)
    
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    
    eval_results[name] = evaluate_model(y_test, preds, probs, model_name=name)
    pred_prob_dict[name] = (preds, probs)

comp_df = compare_models_metrics(eval_results)
print("FINAL TEST SET MODEL COMPARISON TABLE:")
display(comp_df)

## 3. Comparative ROC and Precision-Recall Curves

In [ ]:
fig = plot_roc_pr_curves(pred_prob_dict, y_test.values, save_name=None)
plt.show()

## 4. Confusion Matrices Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, (preds, _)) in enumerate(pred_prob_dict.items()):
    ax = axes[idx]
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["No (0)", "Yes (1)"], yticklabels=["No (0)", "Yes (1)"])
    ax.set_title(f"Confusion Matrix: {name}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()